# Phase 5 — Attributes, Confidence & Archetypes (the ratings engine)

Now that the features pass the football common-sense gate, we build the
actual ratings. Four ideas, applied in order:

1. **Position-normalised percentiles.** Every sub-metric is ranked *within*
   the player's GK/DF/MD/FW peer group. A centre-back is never compared to a
   winger. Rate metrics (accuracy, save %) only count once a minimum volume is
   reached, so "1 of 1" can't masquerade as elite.
2. **Weighted attribute blends → 0–99.** Each of the five attributes is a
   weighted mix of its sub-metrics (e.g. FINISHING = 40 % goals/90 + 30 % shot
   accuracy + …). Goalkeepers swap CREATIVITY→DISTRIBUTION and
   FINISHING→SHOT STOPPING.
3. **Bayesian confidence shrinkage.** Scores are pulled toward the position
   mean for players with few minutes: `w = min/(min+180)`. A great hour of
   football shouldn't outrank a great season.
4. **Deterministic archetypes + an Overall** that is a position-specific
   weighting of the five attributes.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)

career = pd.read_parquet(wl.DATA / "player_career_stats.parquet")
ratings = wl.build_ratings(career)
ratings.to_parquet(wl.DATA / "player_ratings.parquet")
print(f"player_ratings: {ratings.shape}  ({ratings.overall.notna().sum():,} players scored)")

player_ratings: (2803, 108)  (2,803 players scored)


## Overall distribution & range
Percentile-based scores should fill the 1–99 range with a believable spread.

In [2]:
print("Overall — describe:")
print(ratings.overall.describe().round(1).to_string())
print("\nOverall by position (mean):")
print(ratings.groupby("position_code").overall.mean().reindex(wl.POSITIONS).round(1).to_string())

Overall — describe:
count    2803.0
mean       73.0
std         8.5
min        52.0
25%        66.0
50%        73.0
75%        79.0
max        98.0

Overall by position (mean):
position_code
GK    73.0
DF    73.1
MD    72.9
FW    72.9


## Top 20 players overall (≥ 1800 minutes — five-star confidence only)
The headline test: does the Overall leaderboard look like a list of the best
footballers in this data?

In [3]:
proven = ratings[ratings.confidence_stars == 5]
cols = ["short_name", "position_code", "overall", "archetype",
        "passing_score", "creativity_score", "finishing_score",
        "defending_score", "work_rate_score", "confidence_stars"]
top = proven.nlargest(20, "overall")[cols].reset_index(drop=True)
top.index = top.index + 1
top

,short_name,position_code,overall,archetype,passing_score,creativity_score,finishing_score,defending_score,work_rate_score,confidence_stars
1,Neymar,FW,98,Complete Forward,84,95,80,29,90,5
2,L. Messi,FW,98,Complete Forward,87,96,83,22,80,5
3,S. Mustafi,DF,96,Physical Defender,67,58,82,91,89,5
4,P. Pogba,MD,95,Box-to-Box,73,89,67,65,91,5
5,Bartra,DF,95,Physical Defender,67,67,83,85,90,5
6,J. Iličić,FW,95,Complete Forward,81,91,71,42,82,5
7,E. Hazard,FW,94,Creative Forward,87,93,66,21,79,5
8,Philippe Coutinho,FW,94,Creative Forward,81,93,67,49,70,5
9,F. Thauvin,FW,94,Creative Forward,73,92,68,63,75,5
10,G. Bale,FW,94,Complete Forward,71,82,80,63,58,5


## Best XI by position (proven players)

In [4]:
for p in wl.POSITIONS:
    best = proven[proven.position_code == p].nlargest(5, "overall")
    names = [f"{r.short_name} ({r.overall}, {r.archetype})" for _, r in best.iterrows()]
    print(f"{p}: " + "  |  ".join(names))

GK: Alisson (90, All-Round Keeper)  |  R. Zieler (87, All-Round Keeper)  |  Y. Sommer (87, All-Round Keeper)  |  M. Hitz (87, All-Round Keeper)  |  M. ter Stegen (87, All-Round Keeper)
DF: S. Mustafi (96, Physical Defender)  |  Bartra (95, Physical Defender)  |  Nacho Monreal (93, Ball-Playing Defender)  |  L. Koscielny (92, Ball-Playing Defender)  |  Sergio Ramos (92, Ball-Playing Defender)
MD: P. Pogba (95, Box-to-Box)  |  S. Milinković-Savić (93, Box-to-Box)  |  Isco (93, Creative Midfielder)  |  M. Brozović (92, Box-to-Box)  |  K. De Bruyne (91, Creative Midfielder)
FW: Neymar (98, Complete Forward)  |  L. Messi (98, Complete Forward)  |  J. Iličić (95, Complete Forward)  |  E. Hazard (94, Creative Forward)  |  Philippe Coutinho (94, Creative Forward)


## Named ground-truth checks
The validation plan calls out specific players. Do they land where football
knowledge says they should?

In [5]:
def pct_rank_in_pos(short_name, attr):
    row = ratings[ratings.short_name == short_name]
    if row.empty:
        return None
    r = row.iloc[0]
    peers = ratings[ratings.position_code == r.position_code]
    score = r[f"{attr}_score"]
    pctile = (peers[f"{attr}_score"] < score).mean() * 100
    return r.position_code, int(score), round(pctile, 1)

for name, attr in [("K. De Bruyne", "creativity"), ("L. Modrić", "passing"),
                   ("L. Messi", "finishing"), ("Cristiano Ronaldo", "finishing"),
                   ("Sergio Ramos", "defending"), ("J. Oblak", "finishing")]:
    res = pct_rank_in_pos(name, attr)
    if res:
        pos, score, pctile = res
        print(f"{name:20s} {attr:11s} -> score {score:2d}  ({pctile:.0f}th pct among {pos})")

K. De Bruyne         creativity  -> score 96  (100th pct among MD)
L. Modrić            passing     -> score 75  (99th pct among MD)
L. Messi             finishing   -> score 83  (97th pct among FW)
Cristiano Ronaldo    finishing   -> score 78  (93th pct among FW)
Sergio Ramos         defending   -> score 72  (95th pct among DF)
J. Oblak             finishing   -> score 74  (99th pct among GK)


## Archetype distribution
Every player gets a label; no bucket should swallow everyone.

In [6]:
print(ratings.groupby("position_code").archetype.value_counts().to_string())

position_code  archetype            
DF             All-Round Defender       865
               Physical Defender         76
               Defensive Anchor          28
               Ball-Playing Defender     18
FW             All-Round Forward        275
               Creative Forward          89
               Clinical Finisher         80
               Complete Forward          66
               Pressing Forward          56
GK             All-Round Keeper         222
               Commanding Keeper          4
               Shot Stopper               1
MD             All-Round Midfielder     755
               Creative Midfielder      168
               Box-to-Box                56
               Defensive Midfielder      32
               Deep Playmaker            12


## One full identity card (data view)

In [7]:
kdb = ratings[ratings.short_name == "K. De Bruyne"].iloc[0]
labels = wl.GK_ATTR_LABELS if kdb.position_code == "GK" else wl.ATTR_LABELS
print(f"{kdb.short_name}  —  Overall {kdb.overall}  {kdb.position_code}")
print(f"Archetype: {kdb.archetype}   |   Confidence: {'★'*kdb.confidence_stars}{'☆'*(5-kdb.confidence_stars)}")
print(f"Competitions: {kdb.competitions}   Minutes: {kdb.minutes_played:.0f}")
for a in wl.ATTRIBUTES:
    print(f"  {labels[a]:14s} {kdb[f'{a}_score']:2d}")
print(f"Strengths: {kdb.strengths}")
print(f"Areas to improve: {kdb.improvements}")

K. De Bruyne  —  Overall 91  MD
Archetype: Creative Midfielder   |   Confidence: ★★★★★
Competitions: ['england' 'euro_2016' 'world_cup']   Minutes: 4074
  PASSING        71
  CREATIVITY     96
  FINISHING      79
  DEFENDING      44
  WORK RATE      67
Strengths: ['CREATIVITY', 'FINISHING']
Areas to improve: []


Phase 5 complete. `player_ratings.parquet` holds every rated player's five
attribute scores, Overall, confidence, archetype, and strengths/areas — the
payload that the web app and `ratings.json` will serve.